# Identity Studio Colab (Stable)

This notebook is a stable, clean-start launcher.

- clones the correct branch,
- installs required dependencies,
- optionally installs max-quality identity backends,
- launches Gradio with a public URL (`share=True`).

In [ ]:
#@title 1) Configure run
BRANCH = "cursor/identity-studio-ed79"  #@param {type:"string"}
INSTALL_MAX_QUALITY = True  #@param {type:"boolean"}
START_ARCFACE_SCORER = True  #@param {type:"boolean"}
PORT = 7860  #@param {type:"integer"}
SHARE = True  #@param {type:"boolean"}

print("branch:", BRANCH)
print("install max quality:", INSTALL_MAX_QUALITY)
print("start arcface scorer:", START_ARCFACE_SCORER)
print("port:", PORT, "share:", SHARE)

In [ ]:
#@title 2) Clean clone + install base dependencies
import os
import shlex
import subprocess

def run(cmd: str):
    print(f"\n$ {cmd}")
    subprocess.check_call(cmd, shell=True)

run("rm -rf /content/foocus_pro")
run(f"git clone -b {shlex.quote(BRANCH)} https://github.com/sunsideaspect/foocus_pro.git /content/foocus_pro")
os.chdir("/content/foocus_pro")
run("python -m pip install --upgrade pip")
run("pip install -r colab/requirements.txt")
print("\nBase setup complete.")

In [ ]:
#@title 3) Optional: max identity quality backends (roop + ArcFace)
import os
import subprocess

os.chdir('/content/foocus_pro')

if INSTALL_MAX_QUALITY:
    subprocess.check_call('bash colab/setup_max_identity.sh /content/foocus_pro', shell=True)
    if START_ARCFACE_SCORER:
        subprocess.check_call(
            'nohup python -m uvicorn colab.arcface_similarity_server:app --host 0.0.0.0 --port 8890 >/content/arcface.log 2>&1 &',
            shell=True,
        )
        print('ArcFace scorer started on :8890')
    else:
        print('ArcFace scorer start skipped (START_ARCFACE_SCORER=False)')
else:
    print('Max quality setup skipped (INSTALL_MAX_QUALITY=False)')

In [ ]:
#@title 4) Launch Gradio (public link)
import os
import subprocess

os.chdir('/content/foocus_pro')
share_flag = '--share' if SHARE else ''
cmd = f'python -m colab.launch_colab {share_flag} --port {PORT}'.strip()
print('Running:', cmd)
subprocess.check_call(cmd, shell=True)

# Keep this cell running. Public URL appears in output above.

## After launch: recommended UI flow

1. Go to **Photo Job** tab.
2. Set preset to `max_identity_vivid` (or `max_identity_quality`).
3. Click **Apply quality preset**.
4. Upload `Identity reference face`.
5. Click **Generate Photo**.